Setup & Save Fresh Predictions to Disk

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 4 — CELL 1: SETUP + SAVE FRESH PREDICTIONS
# Regenerates Fresh predictions and saves them so the simulation
# can run on BOTH datasets. Standalone — reloads from disk.
# ════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future')
from config import PROCESSED_PATH, MODELS_PATH
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("=" * 60)
print("  PHASE 4 — CELL 1: PREPARING BOTH DATASETS")
print("=" * 60)

# ── KAGGLE predictions already saved in Phase 3 ──────────────────
kaggle_preds = pd.read_csv(str(PROCESSED_PATH / 'test_predictions_kaggle.csv'))
print(f"✅ Kaggle predictions loaded: {list(kaggle_preds.columns)}")

# Reload Kaggle test target to align
kaggle_df = pd.read_csv(str(PROCESSED_PATH / 'demand_features_v2.csv'))
kaggle_df['date'] = pd.to_datetime(kaggle_df['date'])
kaggle_df = kaggle_df.sort_values('date').reset_index(drop=True)
split_k = int(len(kaggle_df) * 0.8)
kaggle_test = kaggle_df.iloc[split_k:].copy().reset_index(drop=True)
kaggle_preds = kaggle_preds.reset_index(drop=True)
kaggle_preds['y_true'] = kaggle_test['sales'].values
kaggle_preds.to_csv(str(PROCESSED_PATH / 'phase4_kaggle_preds.csv'), index=False)
print(f"✅ phase4_kaggle_preds.csv saved ({len(kaggle_preds)} rows)")

# ── FRESH predictions — regenerate from saved models ─────────────
print("\n▶ Regenerating FreshRetailNet predictions...")
fresh_df = pd.read_csv(str(PROCESSED_PATH / 'fresh_features.csv'), nrows=50000)
fresh_df['date'] = pd.to_datetime(fresh_df['date'])
fresh_df = fresh_df.sort_values(['store_id','product_id','date']).reset_index(drop=True)

nf = len(fresh_df); tr_end = int(nf*0.70); vl_end = int(nf*0.85)
train_fr = fresh_df.iloc[:tr_end].copy()
test_fr  = fresh_df.iloc[vl_end:].copy().reset_index(drop=True)

FRESH_FEATS = [
    'sales_lag_1','sales_lag_2','sales_lag_3','sales_lag_7','sales_lag_14',
    'rolling_mean_3','rolling_std_3','rolling_mean_7','rolling_std_7',
    'day_of_week','month','quarter','week_of_year','is_weekend','year',
    'sin_1','cos_1','sin_2','cos_2','discount','holiday_flag',
    'avg_temperature','avg_humidity','avg_wind_level','discount_rolling'
]
FRESH_FEATS = [f for f in FRESH_FEATS if f in fresh_df.columns]
X_tr_fr = train_fr[FRESH_FEATS].fillna(0)
y_tr_fr = train_fr['sales']
X_te_fr = test_fr[FRESH_FEATS].fillna(0)
y_te_fr = test_fr['sales']

fresh_preds = pd.DataFrame({'y_true': y_te_fr.values})

# Naive
fresh_preds['pred_naive'] = test_fr['sales_lag_1'].fillna(y_tr_fr.mean()).values

# Classical (fast, retrain on daily series)
daily_tr = train_fr.groupby('date')['sales'].mean().values
n_test   = len(test_fr)
try:
    ets = ExponentialSmoothing(daily_tr, trend='add', seasonal='add',
            seasonal_periods=7, initialization_method='estimated').fit()
    fresh_preds['pred_ets'] = ets.forecast(1)[0]
except Exception:
    fresh_preds['pred_ets'] = y_tr_fr.mean()
try:
    arima = ARIMA(daily_tr, order=(1,1,1)).fit()
    fresh_preds['pred_arima'] = arima.forecast(1)[0]
except Exception:
    fresh_preds['pred_arima'] = y_tr_fr.mean()
try:
    sarima = SARIMAX(daily_tr, order=(1,1,1), seasonal_order=(1,1,0,7)).fit(disp=False)
    fresh_preds['pred_sarima'] = sarima.forecast(1)[0]
except Exception:
    fresh_preds['pred_sarima'] = fresh_preds['pred_arima']

# Tree models from saved .pkl (if Fresh versions were saved); else retrain quick
import xgboost as xgb, lightgbm as lgb
xgb_fr = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                          random_state=42, n_jobs=-1, verbosity=0)
xgb_fr.fit(X_tr_fr, y_tr_fr, verbose=False)
fresh_preds['pred_xgb'] = xgb_fr.predict(X_te_fr)

lgb_fr = lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                           random_state=42, n_jobs=-1, verbose=-1)
lgb_fr.fit(X_tr_fr, y_tr_fr)
fresh_preds['pred_lgb'] = lgb_fr.predict(X_te_fr)

fresh_preds.to_csv(str(PROCESSED_PATH / 'phase4_fresh_preds.csv'), index=False)
print(f"✅ phase4_fresh_preds.csv saved ({len(fresh_preds)} rows)")
print(f"   Columns: {[c for c in fresh_preds.columns if c.startswith('pred_')]}")
print("\n✅ CELL 1 COMPLETE — both datasets ready for Phase 4")

Full Accuracy Metrics (MAE, RMSE, MAPE, sMAPE, CRPS, WIS)

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 4 — CELL 2: ACCURACY BENCHMARKING (incl. CRPS + WIS)
# ════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import sys
sys.path.append('C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future')
from config import PROCESSED_PATH

print("=" * 60)
print("  PHASE 4 — CELL 2: FORECAST ACCURACY METRICS")
print("=" * 60)

# ── Metric functions ─────────────────────────────────────────────
def mae(y, p):  return np.mean(np.abs(y - p))
def rmse(y, p): return np.sqrt(np.mean((y - p) ** 2))
def mape(y, p):
    m = y != 0
    return np.mean(np.abs((y[m] - p[m]) / y[m])) * 100
def smape(y, p):
    return np.mean(2*np.abs(y-p)/(np.abs(y)+np.abs(p)+1e-8)) * 100

def crps_sample(y, p, sigma):
    # CRPS for a Gaussian predictive distribution N(p, sigma)
    # Closed form (Gneiting & Raftery 2007)
    from scipy.stats import norm
    z = (y - p) / sigma
    return np.mean(sigma * (z*(2*norm.cdf(z)-1) + 2*norm.pdf(z) - 1/np.sqrt(np.pi)))

def wis(y, p, sigma, alphas=(0.1, 0.2, 0.5, 0.8, 0.9)):
    # Weighted Interval Score — approximates CRPS via central prediction intervals
    # Build symmetric intervals from the Gaussian predictive distribution
    from scipy.stats import norm
    K = len(alphas)
    total = np.zeros(len(y))
    for a in alphas:
        # central (1-a) interval → lower/upper quantiles
        lo = p + norm.ppf(a/2)     * sigma
        hi = p + norm.ppf(1 - a/2) * sigma
        width = hi - lo
        penalty_lo = (2/a) * np.maximum(lo - y, 0)
        penalty_hi = (2/a) * np.maximum(y - hi, 0)
        total += (a/2) * (width + penalty_lo + penalty_hi)
    return np.mean(total / (K + 0.5))

# ── Compute for one dataset ──────────────────────────────────────
def benchmark(preds_df, dataset_name):
    y = preds_df['y_true'].values.astype(float)
    pred_cols = [c for c in preds_df.columns if c.startswith('pred_')]
    rows = []
    for col in pred_cols:
        p = preds_df[col].values.astype(float)
        resid_sigma = np.std(y - p) + 1e-6   # spread for prob. metrics
        rows.append({
            'Dataset': dataset_name,
            'Model':   col.replace('pred_', '').upper(),
            'MAE':   round(mae(y, p), 4),
            'RMSE':  round(rmse(y, p), 4),
            'MAPE':  round(mape(y, p), 2),
            'sMAPE': round(smape(y, p), 2),
            'CRPS':  round(crps_sample(y, p, resid_sigma), 4),
            'WIS':   round(wis(y, p, resid_sigma), 4),
        })
    return pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)

kaggle_preds = pd.read_csv(str(PROCESSED_PATH / 'phase4_kaggle_preds.csv'))
fresh_preds  = pd.read_csv(str(PROCESSED_PATH / 'phase4_fresh_preds.csv'))

bench_k = benchmark(kaggle_preds, 'Kaggle')
bench_f = benchmark(fresh_preds,  'FreshRetailNet')

print("\n📦 KAGGLE — Accuracy Benchmark:")
print(bench_k.to_string(index=False))
print("\n🥦 FRESHRETAILNET — Accuracy Benchmark:")
print(bench_f.to_string(index=False))

all_bench = pd.concat([bench_k, bench_f], ignore_index=True)
all_bench.to_csv(str(PROCESSED_PATH / 'phase4_accuracy_benchmark.csv'), index=False)
print("\n✅ phase4_accuracy_benchmark.csv saved")
print("✅ All 6 metrics computed: MAE, RMSE, MAPE, sMAPE, CRPS, WIS")

Diebold-Mariano Significance Test

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 4 — CELL 3: DIEBOLD-MARIANO SIGNIFICANCE TEST
# ════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import itertools
from scipy import stats
import sys
sys.path.append('C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future')
from config import PROCESSED_PATH

print("=" * 60)
print("  PHASE 4 — CELL 3: DIEBOLD-MARIANO TESTS")
print("=" * 60)

def dm_test_all(preds_df, dataset_name):
    y = preds_df['y_true'].values.astype(float)
    pred_cols = [c for c in preds_df.columns if c.startswith('pred_')]
    results = []
    for m1, m2 in itertools.combinations(pred_cols, 2):
        e1 = np.abs(y - preds_df[m1].values)
        e2 = np.abs(y - preds_df[m2].values)
        d  = e1 - e2
        n  = len(d)
        denom = np.std(d, ddof=1) / np.sqrt(n)
        if denom == 0:
            continue
        dm_stat = np.mean(d) / denom
        p_val   = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
        better  = m1 if np.mean(e1) < np.mean(e2) else m2
        results.append({
            'Dataset':  dataset_name,
            'Model 1':  m1.replace('pred_','').upper(),
            'Model 2':  m2.replace('pred_','').upper(),
            'DM Stat':  round(dm_stat, 3),
            'P-Value':  round(p_val, 4),
            'Better':   better.replace('pred_','').upper(),
            'Significant (p<0.05)': 'Yes' if p_val < 0.05 else 'No'
        })
    return pd.DataFrame(results)

kaggle_preds = pd.read_csv(str(PROCESSED_PATH / 'phase4_kaggle_preds.csv'))
fresh_preds  = pd.read_csv(str(PROCESSED_PATH / 'phase4_fresh_preds.csv'))

dm_k = dm_test_all(kaggle_preds, 'Kaggle')
dm_f = dm_test_all(fresh_preds,  'FreshRetailNet')

print("\n📦 KAGGLE — significant pairs only:")
print(dm_k[dm_k['Significant (p<0.05)']=='Yes'].to_string(index=False))

dm_all = pd.concat([dm_k, dm_f], ignore_index=True)
dm_all.to_csv(str(PROCESSED_PATH / 'phase4_diebold_mariano.csv'), index=False)
print(f"\n✅ phase4_diebold_mariano.csv saved ({len(dm_all)} comparisons)")
print("   Confirms whether AI model outperformance is statistically significant")